# BIOSTAT 707 Checkpoint 2

**Author:** Mai Li  
**NetID:** ml783

## AI use statement

I used GPT/Codex as a supporting tool throughout this analysis. After reviewing the assignment requirements and the Checkpoint 1 findings, I guided the tool by specifying the prediction objective, the aspects of the data that required particular attention, and the analyses and comparisons that needed to be included. GPT/Codex helped me clarify statistical concepts, develop and implement parts of the modeling workflow, troubleshoot coding and numerical issues, and improve the presentation of tables, figures, and explanations. I reviewed the proposed methods and code, checked the data exclusions and potential leakage, evaluated the model outputs, and requested revisions when results were unstable or inconsistent with the intended analysis. I made the final decisions about the analysis and interpretation and take responsibility for the submitted work.

## Study objective and analytic cohort

This study examines how well information collected during the first 48 hours of an ICU stay can distinguish patients who die in the hospital from those who survive. The outcome, `In-hospital_death`, is linked to the clinical measurements by `RecordID`. Each ICU stay contributes one row to the analytic data set, with repeated measurements represented by stay-level summaries. This structure supports evaluation at the level of the ICU stay and prevents information from the same stay from appearing in both training and validation data.

The predictors are derived only from information available within the 0–48 hour observation window. `RecordID` is used for linkage rather than prediction, while `Length_of_stay` and `Survival` are excluded because they describe events occurring after the prediction window. SAPS-I and SOFA are kept separate from the fitted models so that they can serve as clinical reference scores.

The data-cleaning rules developed during the initial data assessment are carried forward. Documented `-1` codes, other negative measurements, invalid scale or category values, and zero values for `Height` or `Weight` are treated as missing. Zero height and weight are physiologically implausible, whereas unusual values that do not violate a documented rule are retained rather than removed using unverified clinical cutoffs. Repeated measurements are summarized by their first, last, minimum, maximum, and mean values, together with the number of valid measurements. Numerical summaries missing in more than 90% of stays are excluded because their values cannot be estimated reliably. Measurement-count predictors are retained so that the model can still use information about whether and how often a test was obtained.

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = Path.cwd()
assert (ROOT / 'checkpoint2.ipynb').exists()
subprocess.run(['python', 'scripts/run_analysis.py'], check=True)
subprocess.run(['python', 'scripts/run_checkpoint2.py'], check=True)

In [ ]:
wide = pd.read_csv(ROOT / 'output/set-a_wide.csv')
screening = pd.read_csv(ROOT / 'output/cp2_predictor_screening.csv')
cohort = pd.DataFrame({
    'quantity': ['ICU stays', 'Deaths', 'Death percentage', 'Candidate predictors', 'Retained predictors', 'Dropped for >90% missing'],
    'value': [len(wide), int(wide['In-hospital_death'].sum()), 100 * wide['In-hospital_death'].mean(), len(screening), screening.status.eq('Retained').sum(), screening.status.str.startswith('Dropped').sum()]
})
display(cohort)
display(screening.query("status != 'Retained'").sort_values('missing_proportion', ascending=False))

retained = screening.loc[screening.status.eq('Retained'), 'predictor'].tolist()
baseline = [name for name in ['Age', 'Gender', 'Height', 'Weight', 'ICUType'] if name in retained]
predictor_groups = pd.DataFrame([
    ('Baseline characteristics', ', '.join(baseline), len(baseline), 'Describe patient characteristics and care setting at ICU admission.'),
    ('First observed value', 'Dynamic variables with the `_first` suffix', sum(name.endswith('_first') for name in retained), 'Represents the earliest observed physiologic or laboratory state.'),
    ('Last observed value', 'Dynamic variables with the `_last` suffix', sum(name.endswith('_last') for name in retained), 'Represents the most recent state within the 48-hour window.'),
    ('Minimum value', 'Dynamic variables with the `_min` suffix', sum(name.endswith('_min') for name in retained), 'Captures potentially important low values during the observation window.'),
    ('Maximum value', 'Dynamic variables with the `_max` suffix', sum(name.endswith('_max') for name in retained), 'Captures potentially important high values during the observation window.'),
    ('Mean value', 'Dynamic variables with the `_mean` suffix', sum(name.endswith('_mean') for name in retained), 'Summarizes the typical level across repeated measurements.'),
    ('Measurement count', 'Dynamic variables with the `_count` suffix', sum(name.endswith('_count') for name in retained), 'Reflects measurement availability and monitoring intensity.'),
    ('Total', 'All retained predictors', len(retained), 'Defines the complete candidate input set for elastic net regression and random forest.')
], columns=['predictor_group', 'predictors', 'n_predictors', 'clinical_role'])
display(predictor_groups)

The analytic cohort contains 4,000 ICU stays, including 554 in-hospital deaths (13.85%). The predictor set combines baseline characteristics with complementary summaries of physiologic and laboratory measurements. First and last values describe the patient at the beginning and end of the observation window; minima and maxima capture clinically relevant excursions; means describe the overall level; and measurement counts characterize how intensively a variable was monitored. Fold-specific missingness indicators are also included because the absence of a test may reflect clinical assessment and care processes, although this information should not be interpreted causally. `ICUType` is represented as a categorical variable rather than as a numerical severity scale.

## Model development and internal validation

Model performance is estimated using five-fold stratified cross-validation, with the same held-out stays used for every method. The 90% missingness screen and all preprocessing decisions are estimated within each training fold before being applied to its validation fold. Continuous predictors and the binary `Gender` indicator are imputed with training-fold medians; numerical predictors with missing values also receive fold-specific missingness indicators. `ICUType` is imputed with the training-fold mode and one-hot encoded. In the natural spline model, both `Gender` and `ICUType` are treated as categorical variables and imputed with their training-fold modes. Elastic net standardizes every transformed predictor, including binary, missingness, and `ICUType` indicators; scaling is unnecessary for the tree-based model. The same preprocessing pipeline is refitted within the three-fold inner cross-validation used to select elastic net logistic regression and random forest settings by log loss. This nested structure separates predictor screening and model selection from performance evaluation.

**Model 1 — elastic net logistic regression.** Elastic net combines the sparsity of LASSO with the stabilizing effect of ridge. This balance is useful here because the 48-hour summaries create many correlated predictors: the L1 component can remove weak predictors, while the L2 component reduces arbitrary selection among correlated summaries. It does not impose predefined clinical groups, so selection stability is examined across the outer folds. Inner cross-validation considers $C$ values of 0.01, 0.1, and 1.0 and L1 mixing ratios of 0.1, 0.5, and 0.9. After imputation and categorical encoding, every transformed predictor—including missingness indicators and `ICUType` indicators—is standardized before fitting.

**Model 2 — natural cubic spline logistic regression.** Thirteen continuous predictors were chosen a priori for clinical coverage: age and weight; five vital-sign summaries; two renal-function summaries; two hematologic summaries; glucose; and GCS. Sex and ICU type complete the 15-variable model. Each continuous predictor is represented by a natural cubic spline with four degrees of freedom. The natural-spline constraints produce linear behavior beyond the boundary knots and avoid unstable curvature at the extremes. This specification is fixed rather than tuned.

**Model 3 — random forest.** The inner search considers maximum depth 6 or unlimited; minimum leaf size 5 or 15; and `sqrt(p)` or $0.4p$ candidate features per split. Every forest uses 500 trees and class-balanced bootstrap weights. Continuous and binary predictors are imputed with training-fold medians and supplemented with missingness indicators, while `ICUType` is imputed with the training-fold mode and one-hot encoded. The forest handles the resulting predictors without scaling. No early stopping or outer held-out data are used during fitting.

In [ ]:
tuning = pd.read_csv(ROOT / 'output/cp2_tuning_results.csv')
display(tuning)

The selected settings vary modestly across folds because each training sample contains a different subset of ICU stays. The accompanying inner-validation log loss describes the criterion used to select each configuration before evaluation on the corresponding held-out fold.

## Elastic net coefficients and selection stability

The coefficient plot is based on the selected elastic net configuration refitted on all of set-a. Because every transformed predictor is standardized, coefficient magnitudes describe the change in log odds associated with a one-standard-deviation change and are comparable across predictors. Positive coefficients increase predicted log odds and negative coefficients decrease them, conditional on the other features. Elastic net can set coefficients exactly to zero; selection across outer folds is summarized below to distinguish consistently retained signals from choices that depend on the training sample. Correlated summaries can substitute for one another, so instability does not necessarily imply that the underlying clinical measurement is unimportant.

In [ ]:
display(Image(filename=str(ROOT / 'output/figures/cp2_elastic_net_coefficients.png'), alt='Largest standardized elastic net coefficients'))
coefficients = pd.read_csv(ROOT / 'output/cp2_elastic_net_coefficients.csv')
stability = pd.read_csv(ROOT / 'output/cp2_elastic_net_stability.csv')
display(coefficients.nlargest(15, 'abs_coefficient'))
display(stability.sort_values(['selected_folds', 'selection_rate'], ascending=False).head(25))
always_selected = int(stability.selected_folds.eq(5).sum())
ever_selected = int(stability.selected_folds.gt(0).sum())
display(Markdown(f'Across the five outer training samples, **{always_selected}** transformed predictors were retained in every fitted elastic net model and **{ever_selected}** were retained at least once. Consistent selection suggests a comparatively stable conditional signal, whereas intermittent selection indicates sensitivity to sampling and substitution among correlated summaries.'))

## Non-linear effects

The spline panels report adjusted odds ratios relative to the displayed median reference, with other continuous variables fixed at their medians and categorical variables at their modes. Bands are model-based 95% confidence intervals.

The age relationship is non-linear: estimated risk rises more rapidly at older ages, which is clinically consistent with declining physiologic reserve. The relationship for the last recorded heart rate is approximately U-shaped, suggesting higher risk at both low and high heart rates; uncertainty is greater near the extremes where observations are less dense. The relationship for the last recorded temperature is also non-linear, with increased risk at low temperatures and an upward trend at the febrile end, which is consistent with the risks associated with hypothermia and severe infection. These curves describe adjusted associations used for prediction and should not be interpreted as causal effects of changing a measurement.

In [ ]:
display(Image(filename=str(ROOT / 'output/figures/cp2_spline_effects.png'), alt='Spline partial effects with confidence bands'))
effects = pd.read_csv(ROOT / 'output/cp2_spline_effects.csv')
effect_summary = effects.groupby('feature').agg(reference=('reference', 'first'), minimum_or=('odds_ratio', 'min'), maximum_or=('odds_ratio', 'max')).reset_index()
display(effect_summary)

## Model comparison and final choice

SAPS-I and SOFA provide reference points for judging whether the fitted models improve discrimination beyond established ICU severity scores. Performance is summarized by fold-specific AUROC, its mean, and its sample standard deviation. Each ICU stay receives one out-of-fold prediction from the iteration in which that stay belongs to the held-out fold; the corresponding model is trained and tuned using only the other four folds. Pooling these predictions gives one evaluation prediction per stay without evaluating a fitted model on observations it used for training. The five ROC curves are constructed from these pooled out-of-fold predictions or benchmark scores. AUROC describes the ability to rank higher-risk patients above lower-risk patients; it does not assess calibration. Log loss was therefore used during tuning because it also rewards informative probability estimates.

In [ ]:
comparison = pd.read_csv(ROOT / 'output/cp2_model_comparison.csv').sort_values('mean_auroc', ascending=False)
display(comparison.style.format({column: '{:.3f}' for column in comparison if column != 'model'}))
display(Image(filename=str(ROOT / 'output/figures/cp2_oof_roc.png'), alt='Out-of-fold ROC curves for five methods'))
final_summary = pd.read_csv(ROOT / 'output/cp2_final_model_summary.csv')
best = comparison.iloc[0]
benchmarks = comparison[comparison.model.isin(['SAPS-I', 'SOFA'])]
best_benchmark = benchmarks.iloc[benchmarks.mean_auroc.argmax()]
difference = best.mean_auroc - best_benchmark.mean_auroc
variation = max(best.sd_auroc, best_benchmark.sd_auroc)
display(Markdown(f"The highest mean outer-fold AUROC was **{best.mean_auroc:.3f}** for **{best.model}**. The best benchmark was **{best_benchmark.model}** at **{best_benchmark.mean_auroc:.3f}**. Their difference was **{difference:.3f}**, compared with fold-to-fold SD up to **{variation:.3f}**. Therefore the observed advantage {'exceeded' if difference > variation else 'did not exceed'} the larger fold-to-fold SD. The saved final model is **{final_summary.final_model.iloc[0]}**; the prespecified rule selects the highest mean-AUROC fitted model but prefers elastic net when it is within 0.01 of the best because it provides a sparse, interpretable alternative with similar discrimination."))
display(final_summary)

## Reproducibility and limitations

For reproducibility, `output/cp2_final_model.joblib` stores the final model fitted on all set-a data together with its preprocessing, predictor list, settings, and random seed. `output/cp2_oof_predictions.csv` stores the fold assignment, observed outcome, and out-of-fold predictions or scores for all five methods.

This analysis provides internal validation within set-a; set-b and set-c remain untouched for later independent evaluation. Performance may change in a new cohort because clinical practice, measurement patterns, and case mix can differ. Extreme but documentation-valid source values may influence the summaries, and missingness indicators may capture care processes that do not transfer across hospitals. In addition, AUROC does not establish calibration or clinical utility. These questions require separate assessment using calibration measures, decision-curve analysis, subgroup evaluation, and model-explanation methods.